In [22]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_csv("data/Air_Traffic_Passenger_Statistics.csv")

df.head()

,Activity Period,Activity Period Start Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count,data_as_of,data_loaded_at
0,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM
1,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM
2,199907,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM
3,199907,1999/07/01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM
4,199907,1999/07/01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM


In [24]:
# clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# convert date
df["activity_period"] = pd.to_datetime(df["activity_period"].astype(str), format="%Y%m")
df = df.dropna(subset=["activity_period"])

df["year"] = df["activity_period"].dt.year
df["month"] = df["activity_period"].dt.month

# sort by time for time split
df = df.sort_values("activity_period")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39588 entries, 0 to 39587
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   activity_period              39588 non-null  datetime64[ns]
 1   activity_period_start_date   39588 non-null  object        
 2   operating_airline            39588 non-null  object        
 3   operating_airline_iata_code  39272 non-null  object        
 4   published_airline            39588 non-null  object        
 5   published_airline_iata_code  39272 non-null  object        
 6   geo_summary                  39588 non-null  object        
 7   geo_region                   39588 non-null  object        
 8   activity_type_code           39588 non-null  object        
 9   price_category_code          39588 non-null  object        
 10  terminal                     39588 non-null  object        
 11  boarding_area                39588 non-null  o

In [31]:
# threshold top 25%
threshold = df["passenger_count"].quantile(0.75)

# create new column
df["traffic_level"] = np.where(df["passenger_count"] >= threshold, "High", "Low")

split_date = df["activity_period"].quantile(0.8)

train = df[df["activity_period"] < split_date]
test = df[df["activity_period"] >= split_date]

X_train = train.drop(["traffic_level", "passenger_count", "data_as_of", "data_loaded_at", "activity_period"], axis=1)
y_train = train["traffic_level"]

X_test = test.drop(["traffic_level", "passenger_count", "data_as_of", "data_loaded_at", "activity_period"], axis=1)
y_test = test["traffic_level"]

In [32]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [33]:
df.head()

,activity_period,activity_period_start_date,operating_airline,operating_airline_iata_code,published_airline,published_airline_iata_code,geo_summary,geo_region,activity_type_code,price_category_code,terminal,boarding_area,passenger_count,data_as_of,data_loaded_at,year,month,traffic_level
0,1999-07-01,1999/07/01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM,1999,7,High
101,1999-07-01,1999/07/01,TWA,TW,TWA,TW,Domestic,US,Deplaned,Other,Terminal 1,B,37742,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM,1999,7,High
100,1999-07-01,1999/07/01,"TACA International Airlines, S.A.",TA,"TACA International Airlines, S.A.",TA,International,Central America,Enplaned,Other,Terminal 2,D,4390,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM,1999,7,Low
98,1999-07-01,1999/07/01,Swissair,SR,Swissair,SR,International,Europe,Enplaned,Other,Terminal 2,D,4527,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM,1999,7,Low
97,1999-07-01,1999/07/01,Swissair,SR,Swissair,SR,International,Europe,Deplaned,Other,Terminal 2,D,4973,2026/03/20 01:00:30 PM,2026/03/22 03:02:50 PM,1999,7,Low


In [34]:
df["traffic_level"].value_counts()

traffic_level
Low     29690
High     9898
Name: count, dtype: int64

In [35]:
print(X_train.shape, X_test.shape)

(31580, 722) (8008, 722)


In [36]:
print("Train max date:", train["activity_period"].max())
print("Test min date:", test["activity_period"].min())

Train max date: 2021-08-01 00:00:00
Test min date: 2021-09-01 00:00:00


In [38]:
df["activity_period"].unique()[:10]

<DatetimeArray>
['1999-07-01 00:00:00', '1999-08-01 00:00:00', '1999-09-01 00:00:00',
 '1999-10-01 00:00:00', '1999-11-01 00:00:00', '1999-12-01 00:00:00',
 '2000-01-01 00:00:00', '2000-02-01 00:00:00', '2000-03-01 00:00:00',
 '2000-04-01 00:00:00']
Length: 10, dtype: datetime64[ns]